# config_notebook — shared Phase 3 configuration & helpers

`%run` this at the top of every task notebook so all tasks share one source
of truth for the catalog/schema, tables, endpoints, KB indexes, thresholds
and helper functions.

```python
%run ./config_notebook
```

All tables live under one Unity Catalog namespace
(`31500_atims_dev.atims_taxability` by default). The catalog name starts with
a digit, so it is backtick-quoted everywhere. Rules + thresholds are read from
governed Delta tables (`classification_rules`, `pipeline_config`), not code.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when

# ---------------------------------------------------------------------------
# Widgets (environment parameters)
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "31500_atims_dev", "Unity Catalog")
dbutils.widgets.text("schema",  "atims_taxability", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ---------------------------------------------------------------------------
# Accounting month to process (YYYY-MM). Edit this ONE line to pick the month.
# Deliberately NOT a widget: widget values are sticky per-notebook and were the
# cause of Task 3.1 filtering to an empty month. A plain variable cannot get
# stuck. Must be a month that exists in prediction_ready.accounting_date.
# ---------------------------------------------------------------------------
MONTH = "2022-12"

# Clean up any leftover 'month' widget from earlier runs so it can't confuse you.
try:
    dbutils.widgets.remove("month")
except Exception:
    pass

# ---------------------------------------------------------------------------
# Task 3.4 write-up mode. False (default) = fast deterministic templated
# justifications, NO LLM calls — use this for validation runs and while the
# taxability_matrix is still placeholder. True = generate full Opus audit
# write-ups, but ONLY for rows that need an audit narrative (use-tax reversal,
# COMPLEX disagreement, or ML-decided), not every row.
# ---------------------------------------------------------------------------
GENERATE_LLM_WRITEUPS = False

# Backtick the catalog (it starts with a digit) -> `31500_atims_dev`.atims_taxability.<t>
FQ_SCHEMA = f"`{CATALOG}`.{SCHEMA}"
def _t(name):
    return f"{FQ_SCHEMA}.{name}"

# ---------------------------------------------------------------------------
# Unity Catalog tables (all under one catalog.schema)
# ---------------------------------------------------------------------------
TABLES = {
    # Input (from Phase 2)
    "prediction_ready":     _t("prediction_ready"),
    # Final output (Task 3.7)
    "predictions_out":      _t("non_norad_predictions"),
    # Governed rules + config (created by classification_rules.sql / create_reference_tables)
    "classification_rules": _t("classification_rules"),
    "pipeline_config":      _t("pipeline_config"),
    # Other reference / lookup
    "taxability_matrix":    _t("taxability_matrix"),
    "tax_writeups":         _t("tax_writeups"),
    "category_definitions": _t("category_definitions"),
    "state_service_rules":  _t("state_service_tax_rules"),
    "historical_stats":     _t("historical_stats"),
    # XGBoost label index map (StringIndexer inverse, from Phase 2)
    "xgb_label_index":      _t("xgboost_non_norad_label_index"),
}

# Intermediate task tables (chained between notebook tasks via Delta; same schema)
STAGE = {
    "rules_matched": _t("t31_rules_matched"),
    "rag_matched":   _t("t31_rag_matched"),
    "unmatched":     _t("t31_unmatched"),
    "classified":    _t("t32_classified"),
    "tax_lookup":    _t("t33_tax_lookup"),
    "use_tax":       _t("t33b_use_tax"),
    "tax_writer":    _t("t34_tax_writer"),
    "qa":            _t("t35_qa"),
    "reroute":       _t("t36_reroute"),
}

# ---------------------------------------------------------------------------
# Model Serving + Foundation Model endpoints
# ---------------------------------------------------------------------------
ENDPOINTS = {
    "embedding": "databricks-gte-large-en",
    "xgboost":   "xgboost-non-norad",
    "llm":       "databricks-claude-opus-4-8",
}

# ---------------------------------------------------------------------------
# Vector Search indexes (GTE-embedded)
# ---------------------------------------------------------------------------
VS_ENDPOINT = "atims_vs_endpoint"
VS_INDEXES = {
    "rules":    _t("rules_kb_index"),
    "writeups": _t("tax_writeups_index"),
}

# ---------------------------------------------------------------------------
# Feature column — GTE Large 1024-dim dense embedding (NOT TF-IDF)
# ---------------------------------------------------------------------------
FEATURE_COL = "gte_embedding"

# ---------------------------------------------------------------------------
# Thresholds / behaviour — loaded from pipeline_config (defaults as fallback)
# ---------------------------------------------------------------------------
_DEFAULT_PARAMS = {
    "agreement_high_conf": 0.80,
    "rag_match_min_score": 0.75,
    "max_reroute_passes":  2,
    "vs_num_results":      1,
    "amount_z_flag":       3.0,
}


def _load_params(spark):
    p = dict(_DEFAULT_PARAMS)
    try:
        for r in spark.read.table(TABLES["pipeline_config"]).collect():
            k, v, t = r["config_key"], r["config_value"], r["value_type"]
            if k in p and v is not None:
                p[k] = int(v) if t == "int" else float(v) if t == "float" else v
        print("[config] thresholds loaded from pipeline_config")
    except Exception as e:
        print(f"[config] pipeline_config unavailable ({e}); using default thresholds.")
    return p


PARAMS = _load_params(spark)

# ---------------------------------------------------------------------------
# Downstream notification (use a Secret Scope for the real value)
# ---------------------------------------------------------------------------
try:
    WEBHOOK_URL = dbutils.secrets.get(scope="atims", key="phase3_webhook_url")
except Exception:
    WEBHOOK_URL = ""

## Helper functions

In [ ]:
def load_categories(spark):
    """Label space for the classifier. Prefer category_definitions; if that table
    is missing/empty, FALL BACK to the distinct categories in classification_rules
    so the pipeline still has a valid label set without a separate taxonomy table."""
    try:
        rows = (spark.read.table(TABLES["category_definitions"])
                     .select("cluster").distinct().collect())
        cats = sorted(r["cluster"] for r in rows if r["cluster"])
        if cats:
            return cats
        print("[config] category_definitions empty; deriving labels from classification_rules.")
    except Exception as e:
        print(f"[config] category_definitions unavailable ({e}); deriving from classification_rules.")
    rows = (spark.read.table(TABLES["classification_rules"])
                 .select("category").distinct().collect())
    cats = sorted(r["category"] for r in rows if r["category"])
    if not cats:
        raise ValueError("No categories found in category_definitions or classification_rules.")
    return cats


def sql_label_array(categories):
    escaped = ", ".join("'" + c.replace("'", "''") + "'" for c in categories)
    return f"array({escaped})"


def load_xgb_label_decoder(spark):
    try:
        rows = spark.read.table(TABLES["xgb_label_index"]).collect()
        return {int(r["label_index"]): r["cluster"] for r in rows}
    except Exception as e:
        print(f"[config] xgb_label_index unavailable ({e}); assuming endpoint returns names.")
        return {}


def vs_batch_search(index_name, query_texts, num_results, columns):
    """Batch nearest-neighbour search against a Databricks Vector Search index.

    FAIL-OPEN: if the `databricks-vectorsearch` library is not installed, or the
    endpoint/index does not exist, return a no-match (score 0.0) for every query
    instead of raising. RAG is an optional enhancement on top of the explicit
    rules table — when it is unavailable, those lines simply fall through to the
    ML classifier (Task 3.1) or get an empty prior-examples block (Task 3.4).

    To ENABLE real Vector Search: `%pip install databricks-vectorsearch` on the
    cluster and create VS_ENDPOINT + the indexes named in VS_INDEXES.
    """
    none_hit = lambda: {c: None for c in columns} | {"score": 0.0}
    try:
        from databricks.vector_search.client import VectorSearchClient
        vsc = VectorSearchClient(disable_notice=True)
        index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=index_name)
    except Exception as e:
        print(f"[config] Vector Search unavailable ({type(e).__name__}); RAG disabled -> no-match.")
        return [none_hit() for _ in query_texts]
    out = []
    for q in query_texts:
        try:
            if q is None or str(q).strip() == "":
                out.append(none_hit()); continue
            res = index.similarity_search(query_text=str(q), columns=columns, num_results=num_results)
            rows = (res.get("result", {}).get("data_array")) or []
            if not rows:
                out.append(none_hit()); continue
            top = rows[0]
            rec = {c: top[i] for i, c in enumerate(columns)}
            rec["score"] = float(top[-1])
            out.append(rec)
        except Exception:
            out.append(none_hit())
    return out


def load_active_rules(spark):
    return (spark.read.table(TABLES["classification_rules"])
                 .filter(col("active") == lit(True))
                 .filter(col("effective_from").isNull() | (col("effective_from") <= F.current_date()))
                 .filter(col("effective_to").isNull() | (col("effective_to") >= F.current_date())))


def write_stage(df, table):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table))
    print(f"   → wrote {table}")


def read_stage(spark, table):
    return spark.read.table(table)



def normalize_input(df):
    """Map a raw prediction_ready schema onto the canonical column names the
    Phase 3 pipeline expects, so the same notebooks run on either the synthetic
    sample or the real Phase 1/2 export. Idempotent and case-insensitive.

      mic_cln <- mic_cln | mic                     (classification code, e.g. 'SW')
      amount  <- amount  | tran_amount | invoice_amount   (line $ amount, double)

    Any column already present is left untouched. Missing source columns yield
    NULL rather than an error, so partial schemas still flow through.
    """
    # Lowercase ALL column names first. The source export uses UPPERCASE headers
    # (INVOICE_ID, VENDOR_NAME, STATE, ...) but the whole pipeline references
    # lowercase names. Without this, Task 3.7's case-sensitive column check adds
    # duplicate null columns and corrupts the output table.
    df = df.toDF(*[c.lower() for c in df.columns])

    cols = {c.lower() for c in df.columns}

    if "mic_cln" not in cols:
        if "mic" in cols:
            df = df.withColumn("mic_cln", col("mic").cast("string"))
        else:
            df = df.withColumn("mic_cln", lit(None).cast("string"))

    if "amount" not in cols:
        if "tran_amount" in cols:
            src_amt = F.coalesce(col("tran_amount").cast("double"),
                                 col("invoice_amount").cast("double")) \
                      if "invoice_amount" in cols else col("tran_amount").cast("double")
        elif "invoice_amount" in cols:
            src_amt = col("invoice_amount").cast("double")
        else:
            src_amt = lit(None).cast("double")
        df = df.withColumn("amount", src_amt)

    return df

print(f"config_notebook loaded — namespace=`{CATALOG}`.{SCHEMA}  month={MONTH}")